# BM25 Sparse Embeddings with Endee — SciFact Corpus & Queries

*Create BM25 sparse embeddings for a real retrieval dataset, understand the difference between corpus and query encoding, and store everything in JSONL files ready for indexing.*

This notebook shows:
- How to Use Endee with BM25 embeddings with SciFact corpus and queries from HuggingFace
- How to create **corpus embeddings** using `SparseModel.embed()`
- How to create **query embeddings** using `SparseModel.query_embed()`
- How to save both to `.jsonl` files ready for downstream indexing

**Prerequisites:** `endee-model` and `datasets` installed (see Section 1)

---

## Why Two Separate Functions?

BM25 is asymmetric — it weights **documents** and **queries** differently.

| Function | Used for | BM25 weighting |
|----------|----------|----------------|
| `SparseModel.embed(documents)` | Corpus / documents | Full TF×IDF with document-length normalisation |
| `SparseModel.query_embed(query)` | Search queries | IDF-only — no length normalisation |

Using `.embed()` on a query penalises short text by length normalisation, pushing
all weights toward zero. Using `.query_embed()` on a document omits TF weighting,
producing incorrect scores. Always use the right function for the right side.

## 1. Install Dependencies

In [1]:
!pip install "datasets<3.0.0" endee-model tqdm endee sentence-transformers


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Imports

- `datasets` — loads SciFact from HuggingFace
- `SparseModel` — Endee BM25 wrapper; `.embed()` for corpus, `.query_embed()` for queries
- `tqdm` — progress bar for corpus batching
- `json`, `pathlib` — write and inspect JSONL output files

In [2]:
import json
from pathlib import Path

from datasets import load_dataset
from endee_model import SparseModel
from tqdm import tqdm

print("✅ Imports OK")

/Users/raghunandanvenugopal/Documents/test_hybrid/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK


## 3. Configuration

| Variable | Default | Purpose |
|----------|---------|---------|
| `DATASET_ID` | `BeIR/scifact` | HuggingFace dataset to load |
| `SPARSE_MODEL_ID` | `endee/bm25` | BM25 model identifier |
| `CORPUS_OUTPUT_PATH` | `scifact_corpus.jsonl` | Output file for corpus embeddings |
| `QUERIES_OUTPUT_PATH` | `scifact_queries.jsonl` | Output file for query embeddings |
| `BATCH_SIZE` | `256` | Documents per `.embed()` call |

In [3]:
DATASET_ID          = "BeIR/scifact"
SPARSE_MODEL_ID     = "endee/bm25"

CORPUS_OUTPUT_PATH  = Path("scifact_corpus.jsonl")
QUERIES_OUTPUT_PATH = Path("scifact_queries.jsonl")

BATCH_SIZE = 256   # documents per .embed() call

print(f"Dataset       : {DATASET_ID}")
print(f"Sparse model  : {SPARSE_MODEL_ID}")
print(f"Corpus output : {CORPUS_OUTPUT_PATH}")
print(f"Queries output: {QUERIES_OUTPUT_PATH}")
print(f"Batch size    : {BATCH_SIZE}")

Dataset       : BeIR/scifact
Sparse model  : endee/bm25
Corpus output : scifact_corpus.jsonl
Queries output: scifact_queries.jsonl
Batch size    : 256


## 4. Load SciFact Corpus

SciFact is a scientific fact-checking dataset containing ~5,000 PubMed abstracts.
Each corpus record has three fields:

| Field | Description |
|-------|-------------|
| `_id` | Document identifier |
| `title` | Article title |
| `text` | Abstract text — this is what gets embedded |

The BeIR convention: both `config` and `split` are set to `"corpus"`.

In [4]:
print("Loading SciFact corpus ...")
corpus = load_dataset(DATASET_ID, "corpus", split="corpus")
print(f"✅ Corpus loaded: {len(corpus):,} documents\n")

# Preview first two records
for doc in corpus.select(range(2)):
    print(f"  id    : {doc['_id']}")
    print(f"  title : {doc['title']}")
    print(f"  text  : {doc['text'][:120]}...")
    print()

Loading SciFact corpus ...
✅ Corpus loaded: 5,183 documents

  id    : 4983
  title : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
  text  : Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development a...

  id    : 5836
  title : Induction of myelodysplasia by myeloid-derived suppressor cells.
  text  : Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share biological features of activated ada...



## 5. Load SciFact Queries

SciFact queries are scientific claim statements used to retrieve supporting or
refuting evidence. Each query record has two fields:

| Field | Description |
|-------|-------------|
| `_id` | Query identifier |
| `text` | Claim text — this is what gets embedded |

The BeIR convention: both `config` and `split` are set to `"queries"`.

In [5]:
print("Loading SciFact queries ...")
queries = load_dataset(DATASET_ID, "queries", split="queries")
print(f"✅ Queries loaded: {len(queries):,} queries\n")

# Preview first two records
for q in queries.select(range(2)):
    print(f"  id  : {q['_id']}")
    print(f"  text: {q['text']}")
    print()

Loading SciFact queries ...
✅ Queries loaded: 1,109 queries

  id  : 0
  text: 0-dimensional biomaterials lack inductive properties.

  id  : 2
  text: 1 in 5 million in UK have abnormal PrP positivity.



## 6. Initialise the BM25 Sparse Model

`SparseModel("endee/bm25")` downloads the BM25 vocabulary and precomputed IDF weights
on first use and caches them locally. Subsequent runs load from cache instantly.

In [ ]:
print(f"Loading sparse model: {SPARSE_MODEL_ID} ...")
sparse_model = SparseModel(model_name=SPARSE_MODEL_ID)
print("✅ Sparse model ready")


Loading sparse model: endee/bm25 ...
✅ Sparse model ready


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/raghunandanvenugopal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## What a Sparse Embedding Looks Like

Both `.embed()` and `.query_embed()` return `SparseEmbedding` objects with two numpy arrays:

| Attribute | Type | Meaning |
|-----------|------|---------|
| `.indices` | `ndarray[int]` | Vocabulary token IDs with non-zero BM25 weight |
| `.values` | `ndarray[float]` | BM25 weight for each token ID |

Most of the 30,000+ BM25 vocabulary tokens have weight zero — only the tokens that
actually appear in the text get non-zero entries. The serialised JSONL format used
in this notebook:

```json
{
  "id": "4983",
  "sparse_vector": {
    "indices": [412, 8901, 23445],
    "values":  [0.82, 1.41, 0.67]
  },
  "meta": {"text": "...", "title": "..."}
}
```

## 7. Create Corpus Embeddings — `SparseModel.embed()`

`embed(documents, batch_size)` applies **full BM25 document-side weighting**:
TF × IDF with document-length normalisation. Call it on the abstract text of each
corpus record and write the resulting sparse vector to `scifact_corpus.jsonl`.

In [21]:
# Remove any existing file to avoid appending to stale data
CORPUS_OUTPUT_PATH.unlink(missing_ok=True)

total_written = 0
total_skipped = 0

with open(CORPUS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for start in tqdm(range(0, len(corpus), BATCH_SIZE), desc="Embedding corpus"):
        batch  = corpus[start : start + BATCH_SIZE]

        texts  = batch["text"]  if isinstance(batch, dict) else [r["text"]  for r in batch]
        titles = batch["title"] if isinstance(batch, dict) else [r["title"] for r in batch]
        ids    = batch["_id"]   if isinstance(batch, dict) else [r["_id"]   for r in batch]

        # embed() — document-side BM25: full TF×IDF with length normalisation
        sparse_vecs = list(sparse_model.embed(texts, batch_size=BATCH_SIZE))

        for i, sv in enumerate(sparse_vecs):
            if sv is None or not sv.indices.tolist():
                total_skipped += 1
                continue

            record = {
                "id": ids[i],
                "sparse_vector": {
                    "indices": sv.indices.tolist(),
                    "values":  sv.values.tolist(),
                },
                "meta": {
                    "text":  texts[i],
                    "title": titles[i],
                },
            }
            f.write(json.dumps(record) + "\n")
            total_written += 1

print(f"\n✅ Corpus embeddings saved → {CORPUS_OUTPUT_PATH}")
print(f"   Written : {total_written:,}")
print(f"   Skipped : {total_skipped:,}")

Embedding corpus: 100%|██████████| 21/21 [00:10<00:00,  2.02it/s]


✅ Corpus embeddings saved → scifact_corpus.jsonl
   Written : 5,183
   Skipped : 0


### Inspect a Corpus Embedding

Read the first line back to confirm the file is well-formed and see what a
document-side BM25 sparse vector looks like.

In [22]:
with open(CORPUS_OUTPUT_PATH, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print("Corpus sample")
print(f"  id              : {sample['id']}")
print(f"  title           : {sample['meta']['title']}")
print(f"  text (preview)  : {sample['meta']['text'][:100]}...")
print(f"  non-zero tokens : {len(sample['sparse_vector']['indices'])}")
print(f"  top-5 indices   : {sample['sparse_vector']['indices'][:5]}")
print(f"  top-5 values    : {[round(v, 4) for v in sample['sparse_vector']['values'][:5]]}")

Corpus sample
  id              : 4983
  title           : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.
  text (preview)  : Alterations of the architecture of cerebral white matter in the developing human brain can affect co...
  non-zero tokens : 109
  top-5 indices   : [354307472, 794129062, 242156862, 611385325, 581432272]
  top-5 values    : [1.0887, 1.4566, 1.7527, 1.9759, 1.9759]


## 8. Create Query Embeddings — `SparseModel.query_embed()`

`query_embed(query)` applies **BM25 query-side weighting**: IDF-only, no term-frequency
or document-length normalisation. Pass a single query string and consume the returned
iterator with `next()`.

> **Why not use `.embed()` for queries?**  
> `.embed()` applies document-length normalisation. A short query like
> *"does vitamin D reduce cancer risk"* would be penalised by its low token count,
> pushing all BM25 weights toward zero. `.query_embed()` skips this step and
> produces correct query-side BM25 weights.

In [23]:
# Reload queries if this cell is run independently
try:
    queries
except NameError:
    print("'queries' not in scope — reloading from HuggingFace ...")
    queries = load_dataset(DATASET_ID, "queries", split="queries")
    print(f"Loaded {len(queries):,} queries\n")

# Remove any existing file to avoid appending to stale data
QUERIES_OUTPUT_PATH.unlink(missing_ok=True)

total_written = 0
total_skipped = 0

with open(QUERIES_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in tqdm(queries, desc="Embedding queries"):
        qid  = record["_id"]
        text = record["text"]

        # query_embed() — query-side BM25: IDF-only, no TF or length normalisation
        sv = next(sparse_model.query_embed(text))

        if sv is None or not sv.indices.tolist():
            total_skipped += 1
            continue

        entry = {
            "id": qid,
            "sparse_vector": {
                "indices": sv.indices.tolist(),
                "values":  sv.values.tolist(),
            },
            "meta": {"text": text},
        }
        f.write(json.dumps(entry) + "\n")
        total_written += 1

print(f"\n✅ Query embeddings saved → {QUERIES_OUTPUT_PATH}")
print(f"   Written : {total_written:,}")
print(f"   Skipped : {total_skipped:,}")

Embedding queries: 100%|██████████| 1109/1109 [00:00<00:00, 5747.40it/s]


✅ Query embeddings saved → scifact_queries.jsonl
   Written : 1,109
   Skipped : 0


### Inspect a Query Embedding

Read the first query back. Notice the query sparse vector typically has fewer
non-zero dimensions than a corpus document — queries are shorter, so fewer
vocabulary tokens fire.

In [24]:
with open(QUERIES_OUTPUT_PATH, "r", encoding="utf-8") as f:
    sample_q = json.loads(f.readline())

print("Query sample")
print(f"  id              : {sample_q['id']}")
print(f"  text            : {sample_q['meta']['text']}")
print(f"  non-zero tokens : {len(sample_q['sparse_vector']['indices'])}")
print(f"  indices         : {sample_q['sparse_vector']['indices']}")
print(f"  values          : {[round(v, 4) for v in sample_q['sparse_vector']['values']]}")

Query sample
  id              : 0
  text            : 0-dimensional biomaterials lack inductive properties.
  non-zero tokens : 5
  indices         : [1353987491, 834323496, 1678399785, 1861059290, 449792667]
  values          : [1.0, 1.0, 1.0, 1.0, 1.0]


## 9. Summary Statistics

Verify both output files are complete and compute average sparsity.

In [25]:
def file_stats(path):
    count, total_nnz = 0, 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            total_nnz += len(rec["sparse_vector"]["indices"])
            count += 1
    return count, (total_nnz / count if count else 0)

corpus_count,  corpus_avg   = file_stats(CORPUS_OUTPUT_PATH)
queries_count, queries_avg  = file_stats(QUERIES_OUTPUT_PATH)

print(f"{'File':<30} {'Records':>10} {'Avg non-zero tokens':>22}")
print("-" * 64)
print(f"{str(CORPUS_OUTPUT_PATH):<30} {corpus_count:>10,} {corpus_avg:>22.1f}")
print(f"{str(QUERIES_OUTPUT_PATH):<30} {queries_count:>10,} {queries_avg:>22.1f}")
print()
print("✅ Both JSONL files are ready for indexing into Endee.")

File                              Records    Avg non-zero tokens
----------------------------------------------------------------
scifact_corpus.jsonl                5,183                   87.6
scifact_queries.jsonl               1,109                    8.6

✅ Both JSONL files are ready for indexing into Endee.


In [7]:
from endee_model.sparse.bm25 import bm25_languages

print(bm25_languages())

['arabic', 'danish', 'dutch', 'english', 'finnish', 'french', 'german', 'hungarian', 'italian', 'norwegian', 'portuguese', 'romanian', 'russian', 'spanish', 'swedish']


In [8]:
from endee_model import SparseEmbedding

# Create from a {token_id: weight} dictionary
embedding = SparseEmbedding.from_dict({100: 0.5, 200: 0.8, 300: 1.2})

embedding.as_dict()    # {100: 0.5, 200: 0.8, 300: 1.2}
embedding.as_object()  # {'indices': array([100, 200, 300]), 'values': array([0.5, 0.8, 1.2])}

{'values': array([0.5, 0.8, 1.2], dtype=float32),
 'indices': array([100, 200, 300], dtype=int32)}

---

> **Important — Endee-only embeddings**
>
> The BM25 sparse vectors generated by this notebook are designed to work **exclusively with Endee** as the vector database.
> When creating the index you **must** set `sparse_model="endee_bm25"` so that Endee applies the correct
> server-side IDF weights to match the client-side TF weights stored in the JSONL files.
> Using these embeddings with a different vector database will produce incorrect BM25 scores.


## 10. Connect to Endee & Create Index

A hybrid index is created with two key parameters:

| Parameter | Value | Why |
|-----------|-------|-----|
| `dimension` | `384` | Dense vector size from `all-MiniLM-L6-v2` |
| `space_type` | `"cosine"` | Similarity metric for dense vectors |
| `sparse_model` | `"endee_bm25"` | Tells Endee to apply BM25 server-side IDF weights |

`sparse_model="endee_bm25"` is what ties the client-side TF weights (from `.embed()` / `.query_embed()`)
to the server-side IDF table. Without it, the sparse scores will be meaningless.

> Make sure Endee is running locally on `http://127.0.0.1:8080`.

In [1]:
from endee import Endee

INDEX_NAME = "scifact_bm25"
DENSE_DIM  = 384
SPACE_TYPE = "cosine"

client = Endee()
print("Connected to Endee\n")

# Delete if already exists (clean slate)
try:
    client.delete_index(INDEX_NAME)
    print(f"  Deleted existing index: {INDEX_NAME}")
except Exception:
    pass

# sparse_model="endee_bm25" — required for correct BM25 IDF scoring on the server
client.create_index(
    name=INDEX_NAME,
    dimension=DENSE_DIM,
    space_type=SPACE_TYPE,
    sparse_model="endee_bm25",
)
index = client.get_index(INDEX_NAME)
print(f"  Created index: {INDEX_NAME}")
print("\nIndex ready.")

Connected to Endee

  Created index: scifact_bm25

Index ready.


## 11. Index Corpus from JSONL

Each record from `scifact_corpus.jsonl` is upserted into Endee with:

| Upsert field | Source | Description |
|--------------|--------|-------------|
| `id` | `record["id"]` | Original SciFact document ID |
| `vector` | Encoded from `meta.text` | 384-dim dense vector |
| `sparse_indices` | `record["sparse_vector"]["indices"]` | BM25 token IDs |
| `sparse_values` | `record["sparse_vector"]["values"]` | BM25 TF weights |
| `meta` | `record["meta"]` | Title and text for display |

> For large-scale production use, pre-compute dense vectors alongside sparse vectors
> (as `embedding_creation_bm25.py` does) instead of encoding inline during indexing.

In [2]:
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"])
    from sentence_transformers import SentenceTransformer

UPSERT_BATCH = 1000

print("Loading dense model for hybrid indexing ...")
dense_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Dense model loaded\n")

total_indexed = 0
total_skipped = 0
batch_records = []
batch_texts   = []

def flush_batch(records, texts):
    dense_vecs = dense_model.encode(texts)
    points = []
    for rec, dvec in zip(records, dense_vecs):
        sv = rec["sparse_vector"]
        if not sv["indices"]:
            continue
        points.append({
            "id":             rec["id"],
            "vector":         dvec.tolist(),
            "sparse_indices": sv["indices"],
            "sparse_values":  sv["values"],
            "meta":           rec["meta"],
        })
    index.upsert(points)
    return len(points)

with open(CORPUS_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Indexing corpus"):
        rec = json.loads(line)
        if not rec["sparse_vector"]["indices"]:
            total_skipped += 1
            continue
        batch_records.append(rec)
        batch_texts.append(rec["meta"]["text"])

        if len(batch_records) >= UPSERT_BATCH:
            total_indexed += flush_batch(batch_records, batch_texts)
            batch_records, batch_texts = [], []

if batch_records:
    total_indexed += flush_batch(batch_records, batch_texts)

print(f"\nIndexing complete.")
print(f"  Indexed : {total_indexed:,}")
print(f"  Skipped : {total_skipped:,}")

/opt/miniconda3/envs/tehr/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dense model for hybrid indexing ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7791.89it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dense model loaded



NameError: name 'CORPUS_OUTPUT_PATH' is not defined

In [9]:


model = SparseModel(
    model_name="endee/bm25",
    k=1.5,
    b=0.8,
    language="english",
)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/raghunandanvenugopal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
from endee_model import SparseModel

model = SparseModel(model_name="endee/bm25")

documents = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning enables computers to learn from data",
]

for embedding in model.embed(documents):
    print(embedding.as_dict())  # {token_id: weight, ...}

{771291085: 1.6652867794036865, 741580288: 1.6652867794036865, 1621867415: 1.6652867794036865, 1913189942: 1.6652867794036865, 226376294: 1.6652867794036865, 1312749093: 1.6652867794036865}
{1228389567: 1.6652867794036865, 1644170059: 1.895658016204834, 647928480: 1.6652867794036865, 609543353: 1.6652867794036865, 989116115: 1.6652867794036865}


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/raghunandanvenugopal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
from endee_model import SparseModel

model = SparseModel(model_name="endee/bm25")

documents = ["first document text", "second document text"]

# Returns a generator — iterate to get SparseEmbedding objects
for embedding in model.embed(documents, batch_size=256):
    sparse_dict = embedding.as_dict()       # {int: float}
    sparse_obj  = embedding.as_object()     # {'indices': array, 'values': array}

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/raghunandanvenugopal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
query = "search query text"

for embedding in model.query_embed(query):
    print(embedding.as_dict())

{970674652: 1.0, 553238108: 1.0, 1993922399: 1.0}


In [13]:
count = model.token_count("some text here")
print(f"Token count: {count}")

Token count: 3


In [14]:
from endee_model import SparseModel

model = SparseModel(model_name="endee/bm25")

documents = ["first document text", "second document text"]

# Returns a generator — iterate to get SparseEmbedding objects
for embedding in model.embed(documents, batch_size=256):
    sparse_dict = embedding.as_dict()       # {int: float}
    sparse_obj  = embedding.as_object()     # 

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/raghunandanvenugopal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 12. Search — Hybrid BM25 + Dense Query

Each query uses **both** dense and sparse components:

- **Dense vector** — encodes the query text with `all-MiniLM-L6-v2` (semantic similarity)
- **Sparse vector** — loaded from `scifact_queries.jsonl`, produced by `query_embed()` (lexical BM25 match)

Endee fuses the two scores server-side using its hybrid ranking algorithm.

In [28]:
TOP_K = 5

# Load all queries from the queries JSONL
query_records = []
with open(QUERIES_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        query_records.append(json.loads(line))

print(f"Loaded {len(query_records)} queries\n")
print("-" * 70)

# Run search for the first 3 queries
for qrec in query_records[:3]:
    query_text = qrec["meta"]["text"]
    q_indices  = qrec["sparse_vector"]["indices"]
    q_values   = qrec["sparse_vector"]["values"]

    # Dense query vector
    query_dense_vec = dense_model.encode(query_text).tolist()

    # Hybrid query — dense + BM25 sparse
    hits = index.query(
        vector=query_dense_vec,
        sparse_indices=q_indices,
        sparse_values=q_values,
        top_k=TOP_K,
    )

    print(f"\nQuery [{qrec["id"]}]: {query_text}")
    print(f"{"Rank":<5} {"Doc ID":<10} {"Score":<10} Title / Text preview")
    print("-" * 70)
    for rank, h in enumerate(hits, 1):
        title   = h["meta"].get("title", "")
        preview = h["meta"].get("text", "")[:80]
        print(f"  {rank:<4} {h["id"]:<10} {h["similarity"]:<10.4f} {title or preview}")
    print()

Loaded 1109 queries

----------------------------------------------------------------------

Query [0]: 0-dimensional biomaterials lack inductive properties.
Rank  Doc ID     Score      Title / Text preview
----------------------------------------------------------------------
  1    26071782   0.0082     Latent membrane protein 1 of Epstein–Barr virus coordinately regulates proliferation with control of apoptosis
  2    29638116   0.0082     Complex Tissue and Disease Modeling using hiPSCs.
  3    21257564   0.0081     How long will long-term potentiation last?
  4    4346436    0.0081     Nonlinear Elasticity in Biological Gels
  5    13231899   0.0079     In situ regulation of DC subsets and T cells mediates tumor regression in mice.


Query [2]: 1 in 5 million in UK have abnormal PrP positivity.
Rank  Doc ID     Score      Title / Text preview
----------------------------------------------------------------------
  1    13734012   0.0163     Prevalent abnormal prion protein in huma

## 13. Cleanup — Delete Index

Removes the index from Endee. Safe to skip if you want to keep it for further experiments.

In [ ]:
try:
    client.delete_index(INDEX_NAME)
    print(f"Deleted index: {INDEX_NAME}")
except Exception as e:
    print(f"Could not delete {INDEX_NAME}: {e}")

print("Cleanup complete.")

---

## Output File Format Reference

```json
// Corpus record  →  scifact_corpus.jsonl
{
  "id": "4983",
  "sparse_vector": {
    "indices": [412, 8901, 23445],
    "values":  [0.82, 1.41, 0.67]
  },
  "meta": {
    "text":  "Vitamin D supplementation reduces the risk of ...",
    "title": "Vitamin D and Cancer Prevention"
  }
}

// Query record  →  scifact_queries.jsonl
{
  "id": "1",
  "sparse_vector": {
    "indices": [412, 23445],
    "values":  [1.12, 0.94]
  },
  "meta": {
    "text": "Vitamin D supplementation is beneficial for cancer prevention."
  }
}
```

| Field | Description |
|-------|-------------|
| `id` | Original ID from the dataset |
| `sparse_vector.indices` | Vocabulary token IDs with non-zero BM25 weight |
| `sparse_vector.values` | BM25 weight for each token |
| `meta.text` | Original text (stored for reference) |
| `meta.title` | Document title (corpus records only) |

---

## Key Takeaways

- **Use `.embed()` for corpus documents** — full BM25 TF×IDF with length normalisation.
- **Use `.query_embed()` for queries** — IDF-only, no length penalty.
- **Mixing them gives wrong BM25 scores** — the asymmetry is intentional and important.
- **Output is sparse** — only tokens present in the text get non-zero entries. Documents produce denser vectors than queries because documents are longer.
- **JSONL is the indexing-ready format** — these files load directly into an Endee hybrid index via `index.upsert()`.

---

*Dataset: `BeIR/scifact` via HuggingFace Datasets. Model: `endee/bm25` via `endee-model`.*